# webgpu-dna on Kaggle's free GPU (WGSL via wgpu-py)

Kaggle gives a free CUDA GPU (T4 x2 / P100, ~30 GPU-hrs/week). Our physics is
**WGSL** (WebGPU), not CUDA — so the bridge is [`wgpu-py`](https://github.com/pygfx/wgpu-py),
Python bindings to the same `wgpu-native` (Rust/Dawn) runtime the roadmap's
`webgpu-dna-native` would use. It runs WGSL compute on the GPU through Vulkan.

**This notebook is a PROBE**, not the full simulation. It answers one question:
*does `wgpu-py` actually acquire Kaggle's GPU and run a WGSL compute shader?*
If yes, porting the real `primary.wgsl` host-side (buffers + dispatch) is worth
it and unlocks free GPU hours for the Phase A+B physics. See `FREE_COMPUTE.md`.

## ⚠️ Before running: turn ON the GPU
Kaggle right sidebar → **Settings → Accelerator → GPU T4 x2** (or P100).
Without it, `wgpu-py` falls back to the CPU (llvmpipe) software adapter and
this probe will say so.

In [ ]:
# wgpu-py bundles wgpu-native (no system Vulkan-dev libs needed), but the
# Vulkan *runtime* loader + the NVIDIA ICD must be present to see the GPU.
!pip -q install wgpu numpy
!apt-get -qq install -y libvulkan1 vulkan-tools >/dev/null 2>&1 || true
import wgpu, numpy as np
print('wgpu-py version:', wgpu.__version__)

In [ ]:
# --- Probe 1: which adapter does wgpu-py get? ---
adapter = wgpu.gpu.request_adapter_sync(power_preference='high-performance')
info = dict(adapter.info)
print('Adapter info:')
for k, v in info.items():
    print(f'  {k:16} {v}')

atype = str(info.get('adapter_type', '')).lower()
device_name = str(info.get('device', '') or info.get('description', '')).lower()
on_gpu = ('cpu' not in atype) and ('llvmpipe' not in device_name) and ('software' not in device_name)
print()
if on_gpu:
    print('✅ wgpu-py is on a real GPU adapter:', info.get('device') or info.get('description'))
else:
    print('⚠️  wgpu-py fell back to a CPU/software adapter — enable the Kaggle GPU accelerator')
    print('    (Settings → Accelerator → GPU) and re-run. The probe still works, just slow.')

In [ ]:
# --- Probe 2: run an actual WGSL compute shader and verify the result ---
# Trivial kernel: data[i] = data[i]*2 + 1, in-place over a storage buffer.
from wgpu.utils.compute import compute_with_buffers

WGSL = '''
@group(0) @binding(0) var<storage, read_write> data: array<f32>;
@compute @workgroup_size(64)
fn main(@builtin(global_invocation_id) gid: vec3<u32>) {
  let i = gid.x;
  if (i < arrayLength(&data)) { data[i] = data[i] * 2.0 + 1.0; }
}
'''

n = 4096
data = np.arange(n, dtype=np.float32)
out = compute_with_buffers(
    input_arrays={0: data},
    output_arrays={0: (n, 'f')},
    shader=WGSL,
    n=(n // 64, 1, 1),
)
result = np.frombuffer(out[0], dtype=np.float32)
expected = data * 2.0 + 1.0
ok = np.allclose(result, expected)
print('first 5 in :', data[:5])
print('first 5 out:', result[:5])
print('expected   :', expected[:5])
print()
print('✅ WGSL compute executed correctly on this adapter' if ok else '❌ mismatch — WGSL did not run as expected')

## If both probes pass
`wgpu-py` runs our WGSL on Kaggle's free GPU — the foundation for a Python
`webgpu-dna-native`. The next step is porting the host side of the harness
(`src/gpu/buffers.ts`, `pipelines.ts`, `dispatch.ts` → Python: create the
storage buffers, pack the `params` struct, dispatch `primary.wgsl` one
thread per primary, read back `rad_buf`/dose). The WGSL shaders themselves
port verbatim. The cell below pulls them so you can see what's involved.

In [ ]:
# Pull the actual shaders that would be dispatched (verbatim-portable WGSL).
!git clone --depth 1 -q https://github.com/abgnydn/webgpu-dna /kaggle/working/webgpu-dna || (cd /kaggle/working/webgpu-dna && git pull -q)
import os
root = '/kaggle/working/webgpu-dna'
for f in ['src/shaders/primary.wgsl', 'src/shaders/secondary.wgsl', 'src/shaders/helpers.wgsl', 'public/cross_sections.wgsl']:
    p = os.path.join(root, f)
    if os.path.exists(p):
        print(f'{f:34} {os.path.getsize(p)//1024:5} KB')
print()
print('These compile under wgpu-native unchanged; only the TS host orchestration needs a Python port.')